## A Search for a Puzzle Solver*

**Objective:** Solve the 8-puzzle using A* search.  

**Problem Statement:** The 8-puzzle involves sliding tiles to achieve a goal state. Use A* to solve it.  

**Tasks:**  
Define heuristic functions:  
● H1: Number of misplaced tiles.  
● H2: Sum of Manhattan distances of all tiles from their goal positions.  
● Implement A* with both heuristics.  
● Compare the performance of the two heuristics in terms of the number of nodes explored and solution depth.  

In [4]:
from heapq import heappush, heappop

GOAL = (1, 2, 3,
        4, 5, 6,
        7, 8, 0)   # 0 = blank

GOAL_POS = {v: i for i, v in enumerate(GOAL)}

MOVES = {
    0: (1, 3),
    1: (0, 2, 4),
    2: (1, 5),
    3: (0, 4, 6),
    4: (1, 3, 5, 7),
    5: (2, 4, 8),
    6: (3, 7),
    7: (4, 6, 8),
    8: (5, 7),
}


def h1_misplaced(state):
    return sum(1 for i, v in enumerate(state) if v != 0 and v != GOAL[i])


def h2_manhattan(state):
    dist = 0
    for i, v in enumerate(state):
        if v != 0:
            gi = GOAL_POS[v]
            dist += abs(i // 3 - gi // 3) + abs(i % 3 - gi % 3)
    return dist


def neighbors(state):
    z = state.index(0)
    for nz in MOVES[z]:
        s = list(state)
        s[z], s[nz] = s[nz], s[z]
        yield tuple(s)


def is_solvable(state):
    arr = [x for x in state if x != 0]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv % 2 == 0


def reconstruct_path(parent, goal_state):
    path = []
    cur = goal_state
    while cur is not None:
        path.append(cur)
        cur = parent[cur]
    return path[::-1]


def astar(start, heuristic):
    if not is_solvable(start):
        return None

    g_cost = {start: 0}
    parent = {start: None}
    open_heap = []
    heappush(open_heap, (heuristic(start), 0, start))

    closed = set()
    nodes_explored = 0

    while open_heap:
        _, g, state = heappop(open_heap)
        if state in closed:
            continue

        closed.add(state)
        nodes_explored += 1

        if state == GOAL:
            path = reconstruct_path(parent, state)
            return {
                "path": path,
                "depth": len(path) - 1,
                "nodes_explored": nodes_explored
            }

        for nb in neighbors(state):
            if nb in closed:
                continue
            new_g = g + 1
            if new_g < g_cost.get(nb, float("inf")):
                g_cost[nb] = new_g
                parent[nb] = state
                heappush(open_heap, (new_g + heuristic(nb), new_g, nb))

    return None


def print_state(s):
    for i in range(0, 9, 3):
        print(" ".join("_" if x == 0 else str(x) for x in s[i:i+3]))
    print()


def compare(start):
    print("\nStart State:")
    print_state(start)

    for name, h in [("H1: Misplaced Tiles", h1_misplaced),
                    ("H2: Manhattan Distance", h2_manhattan)]:
        result = astar(start, h)

        if result is None:
            print(f"{name}: Unsolvable\n")
            continue

        print(name)
        print("  Solution Depth :", result["depth"])
        print("  Nodes Explored :", result["nodes_explored"])
        print()


# -------- User Input --------
print("Enter the start state (9 numbers, use 0 for blank):")
start_input = tuple(map(int, input().split()))

if len(start_input) != 9:
    print("Invalid input! Please enter exactly 9 numbers.")
else:
    compare(start_input)


Enter the start state (9 numbers, use 0 for blank):



Start State:
3 1 2
_ 8 7
6 5 4

H1: Misplaced Tiles
  Solution Depth : 23
  Nodes Explored : 13049

H2: Manhattan Distance
  Solution Depth : 23
  Nodes Explored : 1104

